# Stand up a Mosaic AI Vector Search endpoint, create a Delta-Sync index over the Lab 4 chunks table, and prove similarity search returns the right rows

Lab 4 produced a chunks Delta table with the standard RAG schema and pre-computed 1024-dim embeddings from `databricks-gte-large-en`. The PM wants a working similarity search today so they can demo at standup. You have one session: provision the Vector Search endpoint, point a Delta-Sync index at the chunks table, prove three test questions return relevant chunks, then add a new chunk to the source and prove the index picks it up.

The hands-on work:

- Create a Mosaic AI Vector Search endpoint named `labrun-vs-endpoint` of type `STANDARD` (Free Edition does not support `STORAGE_OPTIMIZED`).
- Create a Delta-Sync index `workspace.default.labrun_genai_corpus.chunks_idx` over the Lab 4 chunks table using the SELF_MANAGED_EMBEDDINGS path on the pre-computed `embedding` column.
- Run `index.similarity_search(query_text=..., columns=['chunk_id', 'chunk_text'], num_results=3)` for three test questions.
- Append a sentinel chunk to the source Delta table, trigger `index.sync()`, poll until the sentinel appears, and confirm Delta-Sync is functional end to end.

**Time.** Plan for about 75 minutes wall-clock. Endpoint provisioning is the long pole: budget 10 to 15 minutes for the endpoint to reach ONLINE the first time. Hands-on coding is roughly 10 to 15 minutes; the rest is wait time, which is why the brief and the cert YAML budget a 120-minute session window.

**Cost.** Roughly $0.00 to $0.01 per session. Free Edition swallows the Vector Search endpoint and index storage entirely; the only meter that moves is pay-per-token embedding for the four `similarity_search` calls at query time, which is a fraction of a cent. Free Edition swallows the dollars. The thing this lab eats is your patience while the endpoint provisions; budget 10 to 15 minutes for it the first time.

This lab maps to Databricks GenAI Engineer Associate Domain 4: Assembling and Deploying Applications (~22% of exam weight, provisional). Specifically the task statements on creating and querying a Mosaic AI Vector Search Delta-Sync index, configuring Vector Search for a given solution (endpoint type, pipeline type, embedding strategy), and explaining endpoint-vs-index and Delta-Sync-vs-Direct-Vector-Access.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the Databricks SDK, the Vector Search SDK,
# and the OpenAI-compatible client for embedding the sentinel chunk in
# Task 4. All versions pinned per LAB-CREATION-BLUEPRINT build rules.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 databricks-vectorsearch==0.40 openai==1.54.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Vector Search endpoint names are kebab-case
# (workspace-level resource); index names are three-level UC FQNs (catalog.
# schema.index) because the index is a UC-governed object.

import atexit
import getpass
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
from databricks.vector_search.client import VectorSearchClient
from openai import OpenAI

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "mosaic-ai-vector-search-delta-sync-index"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[4].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_genai_corpus"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"
SOURCE_TABLE = f"{LAB_SCHEMA_FQN}.chunks"  # inherited from Lab 4
INDEX_NAME = f"{LAB_SCHEMA_FQN}.chunks_idx"
ENDPOINT_NAME = "labrun-vs-endpoint"

EMBEDDING_DIM = 1024  # databricks-gte-large-en output dim
EMBEDDING_MODEL = "databricks-gte-large-en"

# Three test questions paired with expected-keyword sets. Each question
# should produce a top-3 result where at least one chunk_text contains one
# of the keywords (case-insensitive). The Lab 4 corpus carries the policy /
# onboarding / image-extracted text needed to satisfy these. Keywords are
# substring matches; the validator is deterministic without depending on
# LLM output.
TEST_QUESTIONS = [
    {
        "question": "How do I sign up for Databricks Free Edition?",
        "keywords": ["free edition", "sign up", "signup", "account"],
    },
    {
        "question": "What does Unity Catalog govern?",
        "keywords": ["unity catalog", "governance", "catalog", "metastore"],
    },
    {
        "question": "What is a Delta table?",
        "keywords": ["delta", "parquet", "transaction log", "acid"],
    },
]

SENTINEL_PHRASE = "labrun sentinel chunk for delta sync proof"

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via current_user.me(),
# resolve the Starter SQL warehouse, build the OpenAI-compatible client for
# pay-per-token embedding, initialize VectorSearchClient, and confirm the
# Lab 4 chunks table is present. This cell must succeed before the manifest
# cell runs anything, per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass(
    "Databricks workspace URL (https://...azuredatabricks.net or .cloud.databricks.com): "
).strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit a SQL statement to the warehouse and return the parsed result.

    Polls statement state up to wait_seconds, raises on FAILED or CANCELED,
    returns a dict {columns: [...], rows: [[...]]} on SUCCEEDED.
    """
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid,
        statement=statement,
        wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)

    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


# OpenAI-compatible client pointed at the Foundation Model API serving
# endpoints. Used only by the Task 4 validator to embed the sentinel phrase
# before inserting it into the source table.
oai = OpenAI(
    api_key=databricks_token,
    base_url=f"{databricks_host}/serving-endpoints",
)

# Vector Search client. disable_notice=True silences the SDK's interactive
# auth prompt because the lab already authenticated with PAT.
vsc = VectorSearchClient(
    workspace_url=databricks_host,
    personal_access_token=databricks_token,
    disable_notice=True,
)

# Lab 4 prerequisite check. Lab 5 inherits the chunks table from Lab 4; if
# the table is missing the student must run Lab 4 first. This is a hard
# precondition, not a soft warning, because the index-create call needs a
# real source table.
try:
    catalog_part, schema_part, table_part = SOURCE_TABLE.split(".")
    src_check = run_sql(
        "SELECT 1 FROM system.information_schema.tables "
        f"WHERE table_catalog = '{catalog_part}' AND table_schema = '{schema_part}' "
        f"AND table_name = '{table_part}'"
    )
    if not src_check["rows"]:
        print(f"BLOCKED: source table {SOURCE_TABLE} not found.")
        print()
        print("Lab 5 reads the chunks corpus that Lab 4 (extract-and-load-multi-")
        print("format-corpus-to-delta) creates. Run Lab 4 to completion first,")
        print("then come back to this lab. Do not skip Lab 4: the chunk_id,")
        print("chunk_text, and 1024-dim embedding columns this lab indexes are")
        print("all produced there.")
        raise SystemExit(1)
except SystemExit:
    raise
except Exception as exc:
    print(f"Failed to verify source table {SOURCE_TABLE}: {exc}")
    raise SystemExit(1)

print(f"Source table confirmed: {SOURCE_TABLE}")

register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order (index first, then
# endpoint; the endpoint cannot be deleted while it owns an index). Free
# Edition has no hourly billing for Vector Search but the one-endpoint cap
# is the binding constraint, so the orphan scan blocks if a prior session
# left an endpoint named labrun-vs-endpoint alive.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="vector_search_index",
        id=INDEX_NAME,
        region="databricks",
        parent=ENDPOINT_NAME,
        cli_delete_command=(
            f"databricks vector-search-indexes delete-index {INDEX_NAME}"
        ),
    ),
    CleanupResource(
        type="vector_search_endpoint",
        id=ENDPOINT_NAME,
        region="databricks",
        cli_delete_command=(
            f"databricks vector-search-endpoints delete-endpoint {ENDPOINT_NAME}"
        ),
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter implementing the labrun-checks CleanupAdapter protocol
    against the Mosaic AI Vector Search SDK plus Unity Catalog via the SQL
    Statement Execution API. Targets the new GenAI-track resource types
    vector_search_index and vector_search_endpoint, plus the carry-over
    uc_managed_table and uc_schema types in case the orphan scan finds
    schema-level leftovers (the Lab 4 corpus stays untouched by this lab's
    cleanup; the types are here only so an orphan match has a handler)."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "vector_search_index":
            try:
                vsc.delete_index(endpoint_name=resource.parent or ENDPOINT_NAME, index_name=rid)
            except Exception as exc:
                msg = str(exc).lower()
                if "does not exist" in msg or "not_found" in msg or "resourcedoesnotexist" in msg:
                    return
                raise
        elif rtype == "vector_search_endpoint":
            try:
                vsc.delete_endpoint(name=rid)
            except Exception as exc:
                msg = str(exc).lower()
                if "does not exist" in msg or "not_found" in msg or "resourcedoesnotexist" in msg:
                    return
                raise
            # Vector Search endpoint deletion is async; poll until the
            # endpoint disappears. Ceiling 600 seconds per RESOURCE-SAFETY-
            # SPEC slow-teardown patterns.
            deadline = time.time() + 600
            while time.time() < deadline:
                try:
                    info = vsc.get_endpoint(name=rid)
                except Exception as exc:
                    msg = str(exc).lower()
                    if "does not exist" in msg or "not_found" in msg or "resourcedoesnotexist" in msg:
                        return
                    raise
                if info is None:
                    return
                time.sleep(10)
        elif rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "vector_search_index":
            try:
                idx = vsc.get_index(
                    endpoint_name=resource.parent or ENDPOINT_NAME,
                    index_name=rid,
                )
                # get_index returns a client wrapper; describe() raises if
                # the underlying index is gone.
                idx.describe()
                return True
            except Exception as exc:
                msg = str(exc).lower()
                if "does not exist" in msg or "not_found" in msg or "resourcedoesnotexist" in msg:
                    return False
                return False
        if rtype == "vector_search_endpoint":
            try:
                info = vsc.get_endpoint(name=rid)
                return info is not None
            except Exception as exc:
                msg = str(exc).lower()
                if "does not exist" in msg or "not_found" in msg or "resourcedoesnotexist" in msg:
                    return False
                return False
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            result = run_sql(sql)
            return len(result["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        """Return Vector Search endpoints and indexes that look like leftovers.

        Vector Search endpoints/indexes carry tags only on Premium; on Free
        Edition the only reliable signal is the labrun-* name prefix the
        cert convention uses, plus the lab's specific index FQN. The scan
        also checks UC for any schema-level tag matches in case Lab 4's
        cleanup did not run; those are surfaced but not auto-deleted by
        this lab.
        """
        arns = []
        # Endpoints: any endpoint whose name starts with labrun- is in scope.
        try:
            endpoint_listing = vsc.list_endpoints()
            endpoint_records = endpoint_listing.get("endpoints", []) if isinstance(endpoint_listing, dict) else []
            for ep in endpoint_records:
                name = ep.get("name") if isinstance(ep, dict) else None
                if not name:
                    continue
                if name.startswith("labrun-"):
                    arns.append(name)
        except Exception:
            pass
        # Indexes: scan within every labrun-* endpoint we can see. The
        # Vector Search SDK only lists indexes under a given endpoint.
        try:
            endpoint_listing = vsc.list_endpoints()
            endpoint_records = endpoint_listing.get("endpoints", []) if isinstance(endpoint_listing, dict) else []
            for ep in endpoint_records:
                ep_name = ep.get("name") if isinstance(ep, dict) else None
                if not ep_name or not ep_name.startswith("labrun-"):
                    continue
                try:
                    idx_listing = vsc.list_indexes(name=ep_name)
                    idx_records = idx_listing.get("vector_indexes", []) if isinstance(idx_listing, dict) else []
                    for idx in idx_records:
                        idx_name = idx.get("name") if isinstance(idx, dict) else None
                        if idx_name:
                            arns.append(idx_name)
                except Exception:
                    continue
        except Exception:
            pass
        return arns

    def exists(self, credentials, arn):
        """Secondary existence check for the tag-audit phase.

        Tag-listed Vector Search names are dotted UC FQNs (indexes) or
        kebab-case endpoint names. Try the index path first (three dots),
        then the endpoint path. Returns False on confirmed-gone, None on
        unknown.
        """
        if arn.count(".") == 2:  # looks like an index FQN
            try:
                vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=arn).describe()
                return True
            except Exception as exc:
                msg = str(exc).lower()
                if "does not exist" in msg or "not_found" in msg or "resourcedoesnotexist" in msg:
                    return False
                return None
        try:
            info = vsc.get_endpoint(name=arn)
            return info is not None
        except Exception as exc:
            msg = str(exc).lower()
            if "does not exist" in msg or "not_found" in msg or "resourcedoesnotexist" in msg:
                return False
            return None


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


# Orphan scan. Free Edition allows exactly one Vector Search endpoint per
# workspace, so a leftover labrun-vs-endpoint from a prior session blocks
# this lab from creating its own. Refuse to continue and print the cleanup
# command per RESOURCE-SAFETY-SPEC Rule 4.
orphans_found = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans_found:
    print(f"BLOCKED: existing labrun- Vector Search objects were found:")
    for arn in orphans_found:
        print("  -", arn)
    print()
    print("Free Edition allows exactly one Vector Search endpoint per workspace.")
    print("These are leftovers from a previous session and will block the new")
    print("endpoint create call. Run the cleanup cell at the bottom of this")
    print("notebook first, or delete them manually:")
    for arn in orphans_found:
        if arn.count(".") == 2:
            print(f"  databricks vector-search-indexes delete-index {arn}")
        else:
            print(f"  databricks vector-search-endpoints delete-endpoint {arn}")
    raise SystemExit(1)

print("No prior Vector Search orphans found. Safe to create resources for this session.")

## Task 1: Create the Vector Search endpoint and wait for ONLINE

A Mosaic AI Vector Search endpoint is the compute that serves similarity_search queries against one or more indexes. Free Edition allows exactly one endpoint per workspace, type `STANDARD` only (`STORAGE_OPTIMIZED` is Premium and is the right pick once you cross roughly 100M embeddings). The endpoint name in this lab is `labrun-vs-endpoint`.

Endpoint provisioning takes 5 to 15 minutes the first time. The cell below makes the create call, then polls `vsc.get_endpoint(...)` until `endpoint_status.state` reads `ONLINE`. The poll ceiling is 1200 seconds (20 minutes) because Free Edition has been observed to take up to that long during high-load periods. Treat the wait as part of the lab; the next markdown cell down is exam-relevant background reading that you can do while you wait.

Tags on Vector Search endpoints are Premium-only as of March 2026. Free Edition does not surface a `tags` parameter on `create_endpoint`; the orphan scan in setup uses the `labrun-` name prefix instead.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the Vector Search endpoint and poll until ONLINE.

# YOUR CODE: Call vsc.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
# to start provisioning. Wrap it so re-running the cell after a partial run
# does not raise: catch the "already exists" error and proceed to the poll.

print(f"Endpoint create call sent for {ENDPOINT_NAME}. Polling for ONLINE...")

deadline = time.time() + 1200
last_state = None
while time.time() < deadline:
    try:
        info = vsc.get_endpoint(name=ENDPOINT_NAME)
    except Exception as exc:
        print(f"  get_endpoint raised: {exc}. Retrying in 15 seconds...")
        time.sleep(15)
        continue
    state = None
    if isinstance(info, dict):
        state = info.get("endpoint_status", {}).get("state")
    if state != last_state:
        elapsed = int(time.time() - (deadline - 1200))
        print(f"  [{elapsed}s elapsed] endpoint_status.state = {state}")
        last_state = state
    if state == "ONLINE":
        print(f"Endpoint {ENDPOINT_NAME} is ONLINE.")
        break
    # Personality copy for the wait, per design-direction.md.
    print("  Asking the Vector Search endpoint to wake up...")
    print("  Standard endpoints on Free Edition typically come online in 5 to 15 minutes.")
    time.sleep(30)
else:
    print("Endpoint did not reach ONLINE within 20 minutes. Check the Vector")
    print("Search page in the workspace UI for the current status; re-run")
    print("this cell to keep polling.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Vector Search endpoint exists, endpoint_type STANDARD, and
# endpoint_status.state ONLINE. Validator does not poll; the task cell
# above already polled. If the student runs this checkpoint while the
# endpoint is still PROVISIONING, the error_reason names the state so
# they know to wait.


def checkpoint_1(session):
    try:
        info = vsc.get_endpoint(name=ENDPOINT_NAME)
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Endpoint {ENDPOINT_NAME} not found. Did vsc.create_endpoint() "
                f"run? Underlying error: {exc}"
            ),
        )
    if not isinstance(info, dict):
        return CheckpointResult(
            status="error",
            error_reason=f"vsc.get_endpoint returned non-dict {type(info).__name__}.",
        )
    endpoint_type = info.get("endpoint_type")
    state = info.get("endpoint_status", {}).get("state")
    if endpoint_type != "STANDARD":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Endpoint {ENDPOINT_NAME} has endpoint_type={endpoint_type!r}; "
                f"expected STANDARD. STORAGE_OPTIMIZED is Premium-only."
            ),
        )
    if state != "ONLINE":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Endpoint {ENDPOINT_NAME} state is {state!r}; expected ONLINE. "
                f"Re-run the Task 1 code cell to keep polling; provisioning "
                f"takes 5 to 15 minutes on Free Edition."
            ),
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint says either the endpoint cannot be found at all (the create call did not run) or it exists but state is something other than `ONLINE` (still provisioning, or it failed). Read the error_reason first. If state is `PROVISIONING`, you are not stuck; you are waiting.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `vsc.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")`. The constant `ENDPOINT_NAME` already holds `labrun-vs-endpoint`. Wrap it in try/except so a re-run that hits "already exists" does not blow up; the surrounding poll loop already handles waiting for ONLINE.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE block in Task 1:

```python
try:
    vsc.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
    print(f"create_endpoint accepted for {ENDPOINT_NAME}")
except Exception as exc:
    msg = str(exc).lower()
    if "already exists" in msg or "resourcealreadyexists" in msg:
        print(f"Endpoint {ENDPOINT_NAME} already exists; reusing it.")
    else:
        raise
```

</details>

**Wallet check.** Still at $0.00. Vector Search endpoint provisioning is free on Free Edition; Free Edition swallows the compute. The endpoint itself does not consume DBU at rest. Your morning coffee cost everything; the bill on this entire session will be cents at most.

## Task 2: Create the Delta-Sync index on the SELF_MANAGED_EMBEDDINGS path

A Vector Search index is the searchable structure that lives inside an endpoint. A Delta-Sync index reads from a Delta source table via Change Data Feed and auto-syncs as the source changes. The lab uses the SELF_MANAGED_EMBEDDINGS path: the source table already holds 1024-dim vectors in its `embedding` column (Lab 4 wrote them with `databricks-gte-large-en`), so the index points at that column instead of asking Mosaic AI to compute embeddings at write time.

The two main pipeline modes:

- `TRIGGERED`: the engineer calls `index.sync()` to update. Used here. Cheapest, latency 30 seconds to a few minutes per sync.
- `CONTINUOUS`: the index auto-syncs as the source changes with no manual call. Higher always-on compute cost.

The exam tests both. The reflection MCQ at the bottom of this notebook covers the trade-off.

Build the index with these exact parameters: `endpoint_name=ENDPOINT_NAME`, `source_table_name=SOURCE_TABLE`, `index_name=INDEX_NAME`, `primary_key='chunk_id'`, `embedding_vector_column='embedding'`, `embedding_dimension=1024`, `pipeline_type='TRIGGERED'`. Then poll `index.describe()['status']['ready']` until it returns True. The first sync is included in the create call so the ready signal also covers the initial sync.

In [ ]:
# NBVAL_SKIP
# Task 2: Create the Delta-Sync index over the Lab 4 chunks table and poll
# until status.ready is True.

# YOUR CODE: Call vsc.create_delta_sync_index(...) with endpoint_name=
# ENDPOINT_NAME, source_table_name=SOURCE_TABLE, index_name=INDEX_NAME,
# primary_key='chunk_id', embedding_vector_column='embedding',
# embedding_dimension=1024, pipeline_type='TRIGGERED'. Wrap in try/except
# so re-running after a partial run reuses the existing index.

print(f"Index create call sent for {INDEX_NAME}. Polling for status.ready=True...")

deadline = time.time() + 900
while time.time() < deadline:
    try:
        idx = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
        described = idx.describe()
    except Exception as exc:
        print(f"  describe raised: {exc}. Retrying in 10 seconds...")
        time.sleep(10)
        continue
    status_block = described.get("status", {}) if isinstance(described, dict) else {}
    ready = status_block.get("ready")
    detailed = status_block.get("detailed_state") or status_block.get("index_state")
    print(f"  status.ready = {ready}, detailed_state = {detailed}")
    if ready is True:
        print(f"Index {INDEX_NAME} is ready.")
        break
    print("  Teaching the Delta-Sync index where the chunks live...")
    time.sleep(20)
else:
    print("Index did not become ready within 15 minutes. Inspect the Vector")
    print("Search page in the workspace UI for the index status.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Delta-Sync index exists with the exact spec the lab requires.
# Validates index_type=DELTA_SYNC, source_table=SOURCE_TABLE, primary_key=
# 'chunk_id', embedding_vector_columns[0].name='embedding', pipeline_type=
# 'TRIGGERED', status.ready=True.


def checkpoint_2(session):
    try:
        idx = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
        described = idx.describe()
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Index {INDEX_NAME} not found under endpoint {ENDPOINT_NAME}. "
                f"Did create_delta_sync_index run? Underlying error: {exc}"
            ),
        )
    if not isinstance(described, dict):
        return CheckpointResult(
            status="error",
            error_reason=f"index.describe() returned non-dict {type(described).__name__}.",
        )

    index_type = described.get("index_type")
    if index_type != "DELTA_SYNC":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Index {INDEX_NAME} has index_type={index_type!r}; expected "
                f"DELTA_SYNC. Direct Vector Access is not supported on Free Edition."
            ),
        )

    spec = described.get("delta_sync_index_spec", {})
    if spec.get("source_table") != SOURCE_TABLE:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Index source_table is {spec.get('source_table')!r}; expected "
                f"{SOURCE_TABLE!r}. Recreate with source_table_name=SOURCE_TABLE."
            ),
        )
    if spec.get("primary_key") != "chunk_id":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Index primary_key is {spec.get('primary_key')!r}; expected "
                f"'chunk_id'. The Lab 4 chunks table uses chunk_id as the PK."
            ),
        )
    if spec.get("pipeline_type") != "TRIGGERED":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Index pipeline_type is {spec.get('pipeline_type')!r}; expected "
                f"'TRIGGERED'. CONTINUOUS works too but costs more; the lab pins "
                f"TRIGGERED for the cheaper happy path."
            ),
        )
    emb_cols = spec.get("embedding_vector_columns") or []
    if not emb_cols or emb_cols[0].get("name") != "embedding":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Index embedding_vector_columns[0].name is "
                f"{emb_cols[0].get('name') if emb_cols else None!r}; expected "
                f"'embedding'. The Lab 4 chunks table holds the pre-computed "
                f"1024-dim vector in the embedding column."
            ),
        )

    status_block = described.get("status", {})
    if status_block.get("ready") is not True:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Index status.ready is {status_block.get('ready')!r}; expected "
                f"True. The first sync may still be running; re-run the Task 2 "
                f"code cell to keep polling."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message names which field is wrong. If `index_type` is missing or wrong, the create call did not run with the Delta-Sync shape. If `embedding_vector_columns` is empty, the create call probably went down the DATABRICKS_MANAGED path (which expects `embedding_source_column`, not `embedding_vector_column`). Read the error_reason carefully.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `vsc.create_delta_sync_index(...)`. Parameters needed: `endpoint_name=ENDPOINT_NAME`, `source_table_name=SOURCE_TABLE`, `index_name=INDEX_NAME`, `primary_key='chunk_id'`, `embedding_vector_column='embedding'`, `embedding_dimension=1024`, `pipeline_type='TRIGGERED'`. Note: SELF_MANAGED uses `embedding_vector_column` (singular, points at the pre-computed vector column); DATABRICKS_MANAGED would use `embedding_source_column` plus `embedding_model_endpoint_name`. If the source table has CDF disabled, the SDK raises a specific error naming `delta.enableChangeDataFeed`; that came from Lab 4.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE block in Task 2:

```python
try:
    vsc.create_delta_sync_index(
        endpoint_name=ENDPOINT_NAME,
        source_table_name=SOURCE_TABLE,
        index_name=INDEX_NAME,
        primary_key="chunk_id",
        embedding_vector_column="embedding",
        embedding_dimension=1024,
        pipeline_type="TRIGGERED",
    )
    print(f"create_delta_sync_index accepted for {INDEX_NAME}")
except Exception as exc:
    msg = str(exc).lower()
    if "already exists" in msg or "resourcealreadyexists" in msg:
        print(f"Index {INDEX_NAME} already exists; reusing it.")
    else:
        raise
```

</details>

**Wallet check.** Still at $0.00. The initial Delta-Sync read against the source table burns a small slice of the daily DBU quota for the warehouse-less serverless Vector Search compute; Free Edition swallows that line item entirely. Storage of 30-ish 1024-dim vectors is bytes, not gigabytes. Your morning coffee still cost more.

## Task 3: Run similarity_search for three test questions

Three questions, top-3 retrieval each. The cell below loops over `TEST_QUESTIONS`, calls `index.similarity_search(query_text=q, columns=['chunk_id', 'chunk_text'], num_results=3)`, and prints the returned chunks. The index needs `USAGE` on an embedding model endpoint to embed the query text at search time; on Free Edition this resolves automatically because the pay-per-token `databricks-gte-large-en` is available to every signed-in user.

The checkpoint runs the same three queries and verifies that the top-3 results for each question contain at least one keyword from the expected-keyword set defined in the constants. Visual inspection in this cell is for you; the checkpoint is the deterministic validation.

In [ ]:
# NBVAL_SKIP
# Task 3: Run three similarity_search calls and print the results.

idx = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

# YOUR CODE: For each entry in TEST_QUESTIONS, call idx.similarity_search(
# query_text=entry['question'], columns=['chunk_id', 'chunk_text'],
# num_results=3) and print the top-3 chunks. The result shape from the SDK
# is a dict with 'result' -> 'data_array' (list of rows, each row matches
# the requested columns plus a similarity score column).

print("Task 3 query loop complete.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: For each of the three test questions, the top-3 results
# from similarity_search contain at least one chunk_text that matches one
# of the expected keywords (case-insensitive substring). The validator
# re-runs the queries independently so it does not depend on whatever the
# student printed.


def _extract_chunk_texts(search_response):
    """Pull chunk_text strings out of the Vector Search SDK response shape.
    The SDK returns {"manifest": {...}, "result": {"data_array": [...]}}
    where each row is a list aligned with the requested columns plus a
    trailing similarity score.
    """
    rows = []
    if isinstance(search_response, dict):
        result_block = search_response.get("result") or {}
        rows = result_block.get("data_array") or []
    texts = []
    for row in rows:
        if isinstance(row, (list, tuple)) and len(row) >= 2:
            # columns=['chunk_id','chunk_text'] -> chunk_text at index 1
            texts.append(str(row[1] or ""))
    return texts


def checkpoint_3(session):
    try:
        index_handle = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=f"Could not load index {INDEX_NAME}: {exc}",
        )

    missed = []
    for entry in TEST_QUESTIONS:
        question = entry["question"]
        keywords = [k.lower() for k in entry["keywords"]]
        try:
            response = index_handle.similarity_search(
                query_text=question,
                columns=["chunk_id", "chunk_text"],
                num_results=3,
            )
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"similarity_search raised for question {question!r}: {exc}"
                ),
            )
        texts = _extract_chunk_texts(response)
        if not texts:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"similarity_search returned zero results for question "
                    f"{question!r}. Is the source table actually populated?"
                ),
            )
        joined = " ".join(texts).lower()
        if not any(kw in joined for kw in keywords):
            missed.append((question, keywords))

    if missed:
        first_q, first_kws = missed[0]
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Top-3 results for {first_q!r} did not contain any of the "
                f"expected keywords {first_kws}. Confirm the Lab 4 corpus is "
                f"populated; the chunk_text values must contain at least one "
                f"of the keywords for each test question."
            ),
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two failure shapes are common here: the call raises (the index handle is wrong, or the columns argument names a column that does not exist), or the call returns results that miss the keyword. Read the error_reason. If it names a specific question, the chunks in your Lab 4 corpus do not cover the topic; that is a Lab 4 problem, not a Lab 5 problem.

</details>

<details><summary>Hint 2 (direction)</summary>

Three calls in a loop. Pattern: `idx.similarity_search(query_text=entry['question'], columns=['chunk_id', 'chunk_text'], num_results=3)` per entry in `TEST_QUESTIONS`. The result is a dict shaped `{"result": {"data_array": [[chunk_id, chunk_text, score], ...]}}`. Print the chunk_id and the first 100 characters of the chunk_text for each row.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the YOUR CODE block in Task 3:

```python
for entry in TEST_QUESTIONS:
    question = entry["question"]
    response = idx.similarity_search(
        query_text=question,
        columns=["chunk_id", "chunk_text"],
        num_results=3,
    )
    print(f"\nQuestion: {question}")
    rows = (response.get("result") or {}).get("data_array") or []
    for row in rows:
        chunk_id, chunk_text = row[0], row[1]
        preview = (chunk_text or "")[:100].replace("\n", " ")
        print(f"  {chunk_id}: {preview}...")
```

</details>

**Wallet check.** Three similarity_search calls, each embeds the query text once against the pay-per-token `databricks-gte-large-en` endpoint. Roughly 50 to 80 tokens per query embedding, at a fraction of a cent per 1k tokens. Total damage so far: well under one cent. Your morning coffee cost a few hundred times more.

## Task 4: Append a sentinel chunk, sync the index, and confirm it appears

This is the proof that Delta-Sync is functional end to end. A new row appended to the source Delta table is picked up by the next `index.sync()` call via Change Data Feed, embedded by the index (or, in this SELF_MANAGED path, read from the row's `embedding` column), and made queryable in similarity_search.

The validator does the work for you. The pattern is non-trivial (embed the sentinel phrase via the FM API, insert into the Delta source, trigger `index.sync()`, poll for the sentinel to appear in similarity_search results, assert top-3 contains it), and the goal of this task is for you to *see* Delta-Sync run, not to write the polling code from scratch. Lab 6 reuses this index without a sentinel append, so the architecture is what carries forward; the mechanics are a one-liner once you know the shape.

Just run the cell below. The validator prints progress as it goes. Polling ceiling is 300 seconds, which matches the documented sync latency upper bound for TRIGGERED Delta-Sync indexes on Free Edition.

In [ ]:
# NBVAL_SKIP
# Task 4: The validator below does the insert + sync + poll work. The
# student just runs this cell. (The Task 4 markdown explains why the
# validator owns the work this time.)

print("Task 4 has no separate task code cell; run the checkpoint cell below.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Insert a sentinel chunk into the source Delta table, trigger
# index.sync(), poll up to 300 seconds for the sentinel chunk_id to appear
# in similarity_search results, assert it is in the top-3. The validator
# owns the full insert + sync + poll cycle so the student sees the
# architecture run; the lab markdown explains why.


def _embed_text(text):
    """Embed text via the Foundation Model API embedding endpoint and return
    a 1024-dim list[float]. Raises on failure."""
    response = oai.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return list(response.data[0].embedding)


def checkpoint_4(session):
    try:
        # Embed the sentinel phrase up front so we can insert a complete row.
        sentinel_vector = _embed_text(SENTINEL_PHRASE)
        if len(sentinel_vector) != EMBEDDING_DIM:
            return CheckpointResult(
                status="error",
                error_reason=(
                    f"Sentinel embedding returned {len(sentinel_vector)} dims; "
                    f"expected {EMBEDDING_DIM}. The embedding model endpoint "
                    f"is misconfigured."
                ),
            )

        sentinel_id = f"labrun-sentinel-{int(time.time())}"
        # Build the INSERT statement. Vector literal goes through the SQL
        # array() constructor so the typed ARRAY<FLOAT> column accepts it.
        vector_sql = "array(" + ",".join(f"CAST({v} AS FLOAT)" for v in sentinel_vector) + ")"
        # Lab 4's chunks table is (chunk_id STRING, doc_uri STRING,
        # chunk_index INT, chunk_text STRING, embedding ARRAY<FLOAT>,
        # source_format STRING, ingested_at TIMESTAMP).
        chunk_text_escaped = SENTINEL_PHRASE.replace("'", "''")
        insert_sql = (
            f"INSERT INTO {SOURCE_TABLE} "
            f"(chunk_id, doc_uri, chunk_index, chunk_text, embedding, "
            f"source_format, ingested_at) VALUES ("
            f"'{sentinel_id}', 'labrun://sentinel', 9999, "
            f"'{chunk_text_escaped}', {vector_sql}, 'sentinel', "
            f"current_timestamp())"
        )
        try:
            run_sql(insert_sql)
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not append sentinel row to {SOURCE_TABLE}: {exc}. "
                    f"The Lab 4 chunks schema may differ from the lab-pinned "
                    f"shape."
                ),
            )
        print(f"Sentinel row inserted: chunk_id={sentinel_id}")

        # Trigger the sync. On TRIGGERED pipelines the call returns fast;
        # the work happens asynchronously.
        index_handle = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
        try:
            index_handle.sync()
            print("index.sync() accepted; polling for the sentinel to appear...")
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"index.sync() raised: {exc}",
            )

        # Poll up to 300 seconds for the sentinel chunk_id to land in the
        # top-3 results for the sentinel phrase. The sentinel phrase is
        # deliberately distinctive so a working sync makes it the closest
        # match in the corpus.
        deadline = time.time() + 300
        while time.time() < deadline:
            try:
                response = index_handle.similarity_search(
                    query_text=SENTINEL_PHRASE,
                    columns=["chunk_id", "chunk_text"],
                    num_results=3,
                )
                rows = (response.get("result") or {}).get("data_array") or []
                hit_ids = [str(row[0]) for row in rows if isinstance(row, (list, tuple))]
                if sentinel_id in hit_ids:
                    print(f"Sentinel found in top-3 after sync. hit_ids={hit_ids}")
                    return CheckpointResult(status="pass")
                print(f"  sentinel not yet in top-3 (saw {hit_ids}); waiting...")
            except Exception as exc:
                print(f"  similarity_search raised mid-poll: {exc}")
            time.sleep(15)

        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Sentinel chunk_id {sentinel_id!r} did not appear in the top-3 "
                f"similarity_search results within 300 seconds. Delta-Sync "
                f"normally lands within 30 seconds to a few minutes; if it "
                f"missed the ceiling the source table CDF may be off or the "
                f"sync trigger was rejected."
            ),
        )
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

There is nothing to write for Task 4: the validator owns the insert + sync + poll cycle. If the checkpoint fails, the error_reason names the layer that broke. Most failures here come from the source-table schema (Lab 4 left it in an unexpected shape) or CDF not being enabled on the source.

</details>

<details><summary>Hint 2 (direction)</summary>

Two common root causes if the sentinel never appears: (a) `delta.enableChangeDataFeed` is `false` on the source table (Lab 4 Checkpoint 4 enables it; if you skipped or re-created the table without it, the index has nothing to read), or (b) the embedding column dimension does not match `EMBEDDING_DIM=1024` (the index rejects rows silently if the vector shape is wrong). Verify with `DESCRIBE EXTENDED workspace.default.labrun_genai_corpus.chunks`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The validator already runs the full pattern; this hint is for the diagnostic case where the source table is in a broken state. To re-enable CDF and re-trigger sync manually:

```python
run_sql(
    f"ALTER TABLE {SOURCE_TABLE} "
    f"SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)
idx = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
idx.sync()
```

Then re-run the Checkpoint 4 cell. If the schema is wrong (chunk_id missing, embedding column not ARRAY<FLOAT> of dim 1024), the fix is in Lab 4, not Lab 5.

</details>

**Wallet check.** Two embedding calls in Task 4 (one for the sentinel insert, one each iteration of the poll loop for the search), each at fractions of a cent. The Delta-Sync index re-read against the source table is free DBU on Free Edition. Cumulative session damage so far: still well under one cent.

## Cleanup

Time to tear down the Vector Search resources. The cell below runs the manifest in reverse-creation order (index first, then endpoint), verifies both are gone, and reports any orphans the tag scan finds. The Lab 4 corpus (`chunks` table, `raw_docs` table, `source_docs` volume, `labrun_genai_corpus` schema) is intentionally NOT torn down here because Lab 6 reads it. The end-of-track cleanup script in `content/qa/batch-cleanup/` handles the corpus when you are done with the cert.

The endpoint delete is async and can take 30 to 120 seconds; the adapter polls until the endpoint disappears. Re-running this cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down vector_search_index then vector_search_endpoint via
# the inline Databricks adapter. No critical (hourly-billed) resources;
# the cap on this lab is the one-endpoint-per-workspace rule, not money.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold a Vector Search endpoint")
print("or index. The next session that tries to create labrun-vs-endpoint will")
print("hit the one-endpoint cap until this is resolved.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: roughly $0.00 to $0.01.** Free Edition swallowed the Vector Search endpoint and index entirely. The meter that moved was pay-per-token embedding: four to six similarity_search calls (three in Task 3, one to several in Task 4 polling) at roughly 50 to 80 tokens each, plus one sentinel embedding in Task 4, at a fraction of a cent per 1k tokens. The Vector Search resources you stood up are gone; the Lab 4 corpus survives so Lab 6 can read it tomorrow.

## Reflection

These are not graded. They are for you.

1. Walk through what Mosaic AI Vector Search does at index creation time when you pass `embedding_vector_column='embedding'` (the SELF_MANAGED path used by this lab) vs when you pass `embedding_source_column='chunk_text'` and `embedding_model_endpoint_name='databricks-gte-large-en'` (the DATABRICKS_MANAGED path). Name the trade-offs in cost, latency, and operational complexity. When would you choose each?

2. Your team's RAG corpus is going to grow from 30 chunks today to 100 million chunks within six months. What architectural changes does that growth force on the endpoint type, the pipeline type, and the embedding strategy?

## Exam-Style Practice

**Question 1 (Domain 4, Vector Search configuration):**

A GenAI Engineer working for an online retailer is improving their search with vector search plus metadata filtering. Searches can total 80 per second, latency is the most critical metric, they do not mind up-front development cost if it improves accuracy without harming latency. The inventory consists of 100 million items nationwide. How should the engineer set this up?

A. Leverage GTE Large embedding model, use standard vector search with hybrid search and reranking turned on.

B. Leverage GTE Large embedding model, use storage-optimized vector search with hybrid search and reranking turned on.

C. Fine-tune a custom embedding model, use standard vector search, keep hybrid search and reranking off.

D. Fine-tune a custom embedding model, use storage-optimized vector search, keep hybrid search and reranking off.

<details><summary>Show answer</summary>

**Correct: D.**

- A is wrong: standard endpoints are NOT designed for 100M items; storage-optimized is the right type at that scale.
- B is wrong: hybrid search plus reranking ON adds latency; the constraint is latency-critical.
- C is wrong: standard endpoint at 100M items is wrong scale.
- D is correct: at 100M items the engineer needs storage-optimized; the up-front fine-tuning is allowed by the scenario (they accept higher dev cost for accuracy); hybrid + reranking OFF preserves latency. This is verbatim the Question 6 pattern from the official sample-question set.

Note: this question is the answer to "configure vector search for a particular solution based on number of embeddings, update frequency, latency, and cost requirements" task statement.

</details>

**Question 2 (Domain 4, Delta-Sync vs Direct Vector Access):**

A GenAI engineer needs a Vector Search index that auto-updates whenever rows are appended to a Delta source table, with no manual sync code in the application. Which Mosaic AI Vector Search configuration is correct?

A. Direct Vector Access index with manual `upsert_data` calls on every source-table append.

B. Delta-Sync index in TRIGGERED mode pointed at the Delta source table.

C. Delta-Sync index in CONTINUOUS mode pointed at the Delta source table.

D. Direct Vector Access index with a Lakeflow Job that calls `upsert_data` on a 5-minute schedule.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: Direct Vector Access requires the engineer to call `upsert_data` manually; "no manual sync code" rules this out.
- B is partially right but TRIGGERED mode requires the engineer to call `index.sync()` to update; "auto-updates" with no manual code is closer to CONTINUOUS.
- C is correct: CONTINUOUS mode automatically syncs the index as the source table changes. No manual code required. The trade-off (higher always-on compute cost) is acceptable in this scenario.
- D is wrong: a scheduled job IS manual sync code; it just runs on a cron.

</details>